# ERA5 Data: Download & Regional Crop

Downloads ERA5 data at 0.25° and saves both a **global** and a **regionally cropped**
version as Aurora-compatible `Batch` NetCDF files (via `Batch.to_netcdf()`).

**Edit the configuration cell** to change the date, region, or output paths.

**Prerequisites:** CDS API key in `$HOME/.cdsapirc` ([setup guide](https://cds.climate.copernicus.eu/how-to-api)).

In [ ]:
# ── Configuration ─────────────────────────────────────────────
YEAR  = "2023"
MONTH = "01"
DAY   = "01"

# Crop region (degrees).  Lon uses [0, 360) convention.
# Default: Eastern North America  (~100°W–60°W, 26°N–50°N)
LAT_MIN = 26.0
LAT_MAX = 50.0
LON_MIN = 260.0   # 360 - 100 = 260
LON_MAX = 300.0   # 360 -  60 = 300

PATCH_SIZE = 4     # Aurora patch size for 0.25° models

from pathlib import Path

DATA_DIR = (
    Path(__file__).resolve().parent.parent / "data" / "era5"
    if "__file__" in dir()
    else Path.cwd().parent / "data" / "era5"
)
DATA_DIR.mkdir(parents=True, exist_ok=True)
print(f"Data directory: {DATA_DIR}")

## 1. Download ERA5

In [ ]:
import cdsapi

c = cdsapi.Client()
date_str = f"{YEAR}-{MONTH}-{DAY}"

# Static variables
if not (DATA_DIR / "static.nc").exists():
    c.retrieve(
        "reanalysis-era5-single-levels",
        {
            "product_type": "reanalysis",
            "variable": ["geopotential", "land_sea_mask", "soil_type"],
            "year": YEAR, "month": MONTH, "day": DAY, "time": "00:00",
            "format": "netcdf",
        },
        str(DATA_DIR / "static.nc"),
    )
print("static.nc ready")

# Surface-level variables (4 time steps)
fname_surf = f"{date_str}-surface-level.nc"
if not (DATA_DIR / fname_surf).exists():
    c.retrieve(
        "reanalysis-era5-single-levels",
        {
            "product_type": "reanalysis",
            "variable": [
                "2m_temperature", "10m_u_component_of_wind",
                "10m_v_component_of_wind", "mean_sea_level_pressure",
            ],
            "year": YEAR, "month": MONTH, "day": DAY,
            "time": ["00:00", "06:00", "12:00", "18:00"],
            "format": "netcdf",
        },
        str(DATA_DIR / fname_surf),
    )
print(f"{fname_surf} ready")

# Atmospheric variables (13 pressure levels, 4 time steps)
fname_atmos = f"{date_str}-atmospheric.nc"
if not (DATA_DIR / fname_atmos).exists():
    c.retrieve(
        "reanalysis-era5-pressure-levels",
        {
            "product_type": "reanalysis",
            "variable": [
                "temperature", "u_component_of_wind", "v_component_of_wind",
                "specific_humidity", "geopotential",
            ],
            "pressure_level": [
                "50", "100", "150", "200", "250", "300",
                "400", "500", "600", "700", "850", "925", "1000",
            ],
            "year": YEAR, "month": MONTH, "day": DAY,
            "time": ["00:00", "06:00", "12:00", "18:00"],
            "format": "netcdf",
        },
        str(DATA_DIR / fname_atmos),
    )
print(f"{fname_atmos} ready")

## 2. Build Global & Cropped Batches

In [ ]:
import numpy as np
import torch
import xarray as xr
from aurora import Batch, Metadata

date_str = f"{YEAR}-{MONTH}-{DAY}"
static_ds = xr.open_dataset(DATA_DIR / "static.nc", engine="netcdf4")
surf_ds   = xr.open_dataset(DATA_DIR / f"{date_str}-surface-level.nc", engine="netcdf4")
atmos_ds  = xr.open_dataset(DATA_DIR / f"{date_str}-atmospheric.nc", engine="netcdf4")


def build_batch(static, surf, atmos):
    """Build an Aurora Batch from xarray Datasets (uses timesteps 0 and 1)."""
    return Batch(
        surf_vars={
            "2t":  torch.from_numpy(surf["t2m"].values[:2][None]),
            "10u": torch.from_numpy(surf["u10"].values[:2][None]),
            "10v": torch.from_numpy(surf["v10"].values[:2][None]),
            "msl": torch.from_numpy(surf["msl"].values[:2][None]),
        },
        static_vars={
            "z":   torch.from_numpy(static["z"].values[0]),
            "slt": torch.from_numpy(static["slt"].values[0]),
            "lsm": torch.from_numpy(static["lsm"].values[0]),
        },
        atmos_vars={
            "t": torch.from_numpy(atmos["t"].values[:2][None]),
            "u": torch.from_numpy(atmos["u"].values[:2][None]),
            "v": torch.from_numpy(atmos["v"].values[:2][None]),
            "q": torch.from_numpy(atmos["q"].values[:2][None]),
            "z": torch.from_numpy(atmos["z"].values[:2][None]),
        },
        metadata=Metadata(
            lat=torch.from_numpy(surf.latitude.values),
            lon=torch.from_numpy(surf.longitude.values),
            time=(surf.valid_time.values.astype("datetime64[s]").tolist()[1],),
            atmos_levels=tuple(int(lev) for lev in atmos.pressure_level.values),
        ),
    )


# ── Global batch ──
global_batch = build_batch(static_ds, surf_ds, atmos_ds)
print(f"Global  – {global_batch.spatial_shape[0]} lat × {global_batch.spatial_shape[1]} lon")

# ── Crop ──
lat_vals = surf_ds.latitude.values   # 90 → -90, strictly decreasing
lon_vals = surf_ds.longitude.values  # 0 → 359.75, strictly increasing

lat_sel = lat_vals[(lat_vals >= LAT_MIN) & (lat_vals <= LAT_MAX)]
lon_sel = lon_vals[(lon_vals >= LON_MIN) & (lon_vals <= LON_MAX)]

# Trim lon to exact multiple of PATCH_SIZE
excess = len(lon_sel) % PATCH_SIZE
if excess:
    lon_sel = lon_sel[:-excess]

# Trim lat so len % PATCH_SIZE ∈ {0, 1}  (Batch.crop handles the 1 case)
excess = len(lat_sel) % PATCH_SIZE
if excess > 1:
    lat_sel = lat_sel[:-(excess - 1)]

crop_static = static_ds.sel(latitude=lat_sel, longitude=lon_sel)
crop_surf   = surf_ds.sel(latitude=lat_sel, longitude=lon_sel)
crop_atmos  = atmos_ds.sel(latitude=lat_sel, longitude=lon_sel)

cropped_batch = build_batch(crop_static, crop_surf, crop_atmos)
print(
    f"Cropped – {cropped_batch.spatial_shape[0]} lat × {cropped_batch.spatial_shape[1]} lon\n"
    f"          [{lat_sel[-1]:.2f}°N .. {lat_sel[0]:.2f}°N] × "
    f"[{lon_sel[0]:.2f}°E .. {lon_sel[-1]:.2f}°E]"
)

## 3. Save Both Batches

In [ ]:
out_dir = DATA_DIR / "batches"
out_dir.mkdir(exist_ok=True)

global_path  = out_dir / f"{date_str}-global.nc"
cropped_path = out_dir / f"{date_str}-cropped.nc"

global_batch.to_netcdf(global_path)
cropped_batch.to_netcdf(cropped_path)

print(f"Saved {global_path.name}  ({global_path.stat().st_size / 1e6:.1f} MB)")
print(f"Saved {cropped_path.name} ({cropped_path.stat().st_size / 1e6:.1f} MB)")

In [ ]:
# Quick reload check
g = Batch.from_netcdf(global_path)
cr = Batch.from_netcdf(cropped_path)

print(f"Global  reloaded: {g.spatial_shape}, levels={len(g.metadata.atmos_levels)}")
print(f"Cropped reloaded: {cr.spatial_shape}, levels={len(cr.metadata.atmos_levels)}")
print(f"  lat: [{cr.metadata.lat[-1]:.2f}, {cr.metadata.lat[0]:.2f}]")
print(f"  lon: [{cr.metadata.lon[0]:.2f}, {cr.metadata.lon[-1]:.2f}]")